# Notebook 06 -- Diseño del Gating para el Sistema Mixture-of-Experts

---


## Objetivo

Diseñar y comparar 5 mecanismos de routing: hard, soft (distancia), soft (probabilístico), adaptativo, y seleccionar el mejor. Optimizar trade-off entre complejidad y rendimiento.


## Entrada

features + asignaciones de cluster + clasificadores + modelos expertos


## Salida

Gating mechanism seleccionado (hard/soft/adaptativo) + hiperparámetros optimizados


---
## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

---
## Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS = -1

# Parámetros específicos del gating (desde configs/gating.yaml)
THRESHOLD_SEARCH = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
TAU_SEARCH = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
MIN_PROB = 0.05  # probabilidad conjunta mínima para incluir un experto en el gating suave

import os, warnings
warnings.filterwarnings("ignore")

for d in [DATA_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/gating", exist_ok=True)

print("Configuración cargada.")
print(f"  DATA_DIR    = {DATA_DIR}")
print(f"  MODELS_DIR  = {MODELS_DIR}")
print(f"  RESULTS_DIR = {RESULTS_DIR}")

---
## Cargar Variables, Asignaciones de Clúster, Clasificadores y Modelos Expertos

Cargamos todos los artefactos producidos por los Notebooks 02-05:

- **Variables** (`data/features.parquet`): la matriz de variables con
  variables de calendario, rezagos y ventanas móviles.
- **Asignaciones de clúster** (`data/clustering/assignments.parquet`): cada fecha
  mapeada a `(type_day, type_week)`.
- **Centroides** (`data/clustering/centroids_day.npy`, `centroids_week.npy`):
  usados para el gating suave basado en distancia.
- **Clasificadores** (`models/classifiers/classifier_day.pkl`, `models/classifiers/classifier_week.pkl`):
  entrenados en el Notebook 04 con capacidad de `predict_proba`.
- **Modelos expertos** (`models/experts/expert_day{i}_week{j}.pkl`): uno por cada
  combinación de clúster, entrenados en el Notebook 05.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import softmax as scipy_softmax

np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------
# 3a. Cargar matriz de variables
# ------------------------------------------------------------------
features_path = Path(os.path.join(DATA_DIR, "processed", "features.parquet"))
if features_path.exists():
    df_features = pd.read_parquet(features_path)
    # Asegurar que 'datetime' sea una columna y no un índice
    if 'datetime' in df_features.index.names or df_features.index.name == 'datetime':
        df_features = df_features.reset_index()
    print(f"Variables cargadas: {df_features.shape}")
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {features_path}. Ejecuta antes el notebook 02 (make pipeline).")

In [ ]:
df_features.head()

In [ ]:
# ------------------------------------------------------------------
# 3b. Cargar o generar asignaciones de clúster
# ------------------------------------------------------------------
clustering_dir = Path(DATA_DIR) / "clustering"
assignments_path = clustering_dir / "cluster_assignments.parquet"

if assignments_path.exists():
    df_assignments = pd.read_parquet(assignments_path)
    # Renombrar columnas para que coincidan con 'type_day' y 'type_week'
    if 'cluster_day' in df_assignments.columns and 'cluster_week' in df_assignments.columns:
        df_assignments = df_assignments.rename(columns={
            'cluster_day': 'type_day',
            'cluster_week': 'type_week'
        })
    print(f"Asignaciones de clúster cargadas: {df_assignments.shape}")
    # Impresión de diagnóstico para revisar columnas después de cargar/renombrar
    print(f"  Columnas de df_assignments después de cargar/renombrar: {df_assignments.columns.tolist()}")
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {assignments_path}. Ejecuta antes el notebook 03 (make pipeline).")

# Unir etiquetas de clúster a las variables si aún no están presentes
if "type_day" not in df_features.columns:
    if "date" not in df_features.columns:
        df_features["date"] = df_features["datetime"].dt.date
    df_features = df_features.merge(
        df_assignments[["datetime", "type_day", "type_week"]],
        on="datetime", how="left"
    )

# Eliminar NaNs después de que todas las uniones y creaciones de variables estén completas
df_features = df_features.dropna().reset_index(drop=True)

# Identificar combinaciones de clúster
day_labels = sorted(df_features["type_day"].unique())
week_labels = sorted(df_features["type_week"].unique())
cluster_combos = [(d, w) for d in day_labels for w in week_labels]
print(f"\nCombinaciones de clúster (type_day x type_week): {len(cluster_combos)}")
for d, w in cluster_combos:
    n = len(df_features[(df_features["type_day"] == d) & (df_features["type_week"] == w)])
    print(f"  day={d}, week={w}: {n:,} observaciones")

In [ ]:
df_features.head()

In [ ]:
df_features = df_features[~df_features.index.year.isin([2020, 2021])]

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df_features.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
len(calendar_lags)

In [ ]:
import pickle
from pathlib import Path

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

clf_day_path = Path(MODELS_DIR) / "classifiers" / "best_classifier_day.pkl"
clf_week_path = Path(MODELS_DIR) / "classifiers" / "best_classifier_week.pkl"

# Definir columnas de variables (todo excepto objetivo y metadatos)
META_COLS = ["datetime", "date", "demand", "type_day", "type_week"]
# Corregido para usar calendar_lags según la entrada del usuario y el análisis del error
FEATURE_COLS = calendar_lags
print(f"\nColumnas de variables ({len(FEATURE_COLS)}): {FEATURE_COLS[:10]}...")

X_all = df_features[FEATURE_COLS].values
y_all = df_features["demand"].values
y_day = df_features["type_day"].values.astype(int)
y_week = df_features["type_week"].values.astype(int)

# Separación train/test (temporal: último 20% para prueba)
split_idx = int(len(df_features) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
y_day_train, y_day_test = y_day[:split_idx], y_day[split_idx:]
y_week_train, y_week_test = y_week[:split_idx], y_week[split_idx:]

df_test = df_features.iloc[split_idx:].copy().reset_index(drop=True)
print(f"\nEntrenamiento: {X_train.shape[0]:,} | Prueba: {X_test.shape[0]:,}")

if clf_day_path.exists() and clf_week_path.exists():
    with open(clf_day_path, "rb") as f:
        loaded_day_dict = pickle.load(f)
    with open(clf_week_path, "rb") as f:
        loaded_week_dict = pickle.load(f)
    # Extraer el objeto pipeline del diccionario cargado
    classifier_day = loaded_day_dict['pipeline']
    classifier_week = loaded_week_dict['pipeline']

    # Verificar incompatibilidad de variables y forzar reentrenamiento si aplica
    if hasattr(classifier_day, 'steps') and len(classifier_day.steps) > 0 and hasattr(classifier_day.steps[0][1], 'n_features_in_'):
        if classifier_day.steps[0][1].n_features_in_ != len(FEATURE_COLS):
            print("[WARNING] El classifier_day cargado espera un número diferente de variables. Se forzará el reentrenamiento.")
            classifier_day = None  # Invalidar modelo cargado
    if hasattr(classifier_week, 'steps') and len(classifier_week.steps) > 0 and hasattr(classifier_week.steps[0][1], 'n_features_in_'):
        if classifier_week.steps[0][1].n_features_in_ != len(FEATURE_COLS):
            print("[WARNING] El classifier_week cargado espera un número diferente de variables. Se forzará el reentrenamiento.")
            classifier_week = None  # Invalidar modelo cargado

    if classifier_day is not None and classifier_week is not None:
        print("Clasificadores cargados desde disco.")

In [ ]:
# ------------------------------------------------------------------
# 3d. Cargar o entrenar modelos expertos
# ------------------------------------------------------------------
from sklearn.ensemble import GradientBoostingRegressor

experts_dir = Path(MODELS_DIR) / "experts"
os.makedirs(experts_dir, exist_ok=True)

experts = {}
existing_experts = list(experts_dir.glob("expert_day*_week*.pkl"))

if existing_experts:
    for path in existing_experts:
        name = path.stem
        parts = name.replace("expert_day", "").split("_week")
        cd, cw = int(parts[0]), int(parts[1])
        with open(path, "rb") as f:
            experts[(cd, cw)] = pickle.load(f)
    print(f"\nExpertos cargados desde disco: {len(experts)} modelos")
else:
    print("\n[INFO] No se encontraron modelos expertos. Entrenando expertos con GradientBoosting.")
    for d_label in day_labels:
        for w_label in week_labels:
            mask_train = (y_day_train == d_label) & (y_week_train == w_label)
            n_train_cluster = mask_train.sum()
            if n_train_cluster < 30:
                print(f"  Omitiendo day={d_label}, week={w_label}: solo {n_train_cluster} observaciones de entrenamiento")
                continue

            model = GradientBoostingRegressor(
                n_estimators=200, max_depth=5, learning_rate=0.1,
                random_state=RANDOM_SEED
            )
            model.fit(X_train[mask_train], y_train[mask_train])
            experts[(d_label, w_label)] = model

            # Guardar
            path = experts_dir / f"expert_day{d_label}_week{w_label}.pkl"
            with open(path, "wb") as f:
                pickle.dump(model, f)
            print(f"  Experto day={d_label}, week={w_label}: entrenado con {n_train_cluster:,} observaciones")

    print(f"  Total de expertos entrenados: {len(experts)}")

print(f"\nClaves de expertos disponibles: {sorted(experts.keys())}")

---
## Funciones de Métricas

Todas las métricas se definen en línea para que el notebook sea completamente autocontenido.

In [ ]:
# ============================================================
# Métricas — 100% autocontenido, sin imports externos
# ============================================================

def mape(y_true, y_pred):
    """Error Porcentual Absoluto Medio."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def rmse(y_true, y_pred):
    """Raíz del Error Cuadrático Medio."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_metric(y_true, y_pred):
    """Error Absoluto Medio."""
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def compute_all_metrics(y_true, y_pred):
    """Calcula todas las métricas principales y las retorna como diccionario."""
    return {
        "MAPE (%)": mape(y_true, y_pred),
        "RMSE (kWh)": rmse(y_true, y_pred),
        "MAE (kWh)": mae_metric(y_true, y_pred),
    }


print("Funciones de métricas definidas.")

---
## Configuración Compartida de Visualización

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración con calidad de publicación
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("Set2", 8)
METHOD_COLORS = {
    "Hard Gating": PALETTE[0],
    "Soft (Distance)": PALETTE[1],
    "Soft (Probability)": PALETTE[2],
    "Adaptive": PALETTE[3],
}

DAY_TYPE_NAMES = {0: "Normal weekday", 1: "Weekend", 2: "Holiday", 3: "Bridge/Atypical"}

print("Configuración de visualización lista.")

---
## 1. Gating Hard (Baseline Obligatorio)

El gating hard es la estrategia más simple: cada observación se clasifica en exactamente
una combinación de clúster `(type_day, type_week)` y se asigna al experto correspondiente.
Esto es equivalente a un pipeline jerárquico estándar de clasificación seguida de regresión.

**Ventajas**: Simple, rápido, determinista.

**Desventaja**: No mitiga errores de clasificación. Si el clasificador asigna
una observación al clúster equivocado, la predicción utiliza un experto inapropiado
sin posibilidad de corrección.

In [ ]:
X_all = df_features[FEATURE_COLS].values
y_all = df_features["demand"].values
y_day = df_features["type_day"].values.astype(int)
y_week = df_features["type_week"].values.astype(int)

# Train/test split (temporal: ultimos 20% para test)
split_idx = int(len(df_features) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
y_day_train, y_day_test = y_day[:split_idx], y_day[split_idx:]
y_week_train, y_week_test = y_week[:split_idx], y_week[split_idx:]

In [ ]:
# ============================================================
# Gating hard — clasificar, seleccionar un único experto y predecir
# ============================================================

def predict_hard_gating(X, classifier_day, classifier_week, experts):
    """Gating hard: un experto por observación según la salida del clasificador.

    Parámetros
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    classifier_day, classifier_week : clasificadores sklearn ya entrenados
    experts : dict[(int, int), model]

    Retorna
    -------
    predictions : np.ndarray, shape (n_samples,)
    metadata : dict con información de enrutamiento
    """
    pred_day = classifier_day.predict(X)
    pred_week = classifier_week.predict(X)
    predictions = np.full(len(X), np.nan)
    expert_used = []

    for i in range(len(X)):
        key = (int(pred_day[i]), int(pred_week[i]))
        if key in experts:
            predictions[i] = experts[key].predict(X[[i]])[0]
            expert_used.append(key)
        else:
            # Respaldo: promedio de todos los expertos
            preds_all = [exp.predict(X[[i]])[0] for exp in experts.values()]
            predictions[i] = np.mean(preds_all)
            expert_used.append("fallback")

    n_fallback = sum(1 for e in expert_used if e == "fallback")
    return predictions, {
        "expert_used": expert_used,
        "n_fallback": n_fallback,
        "pred_day": pred_day,
        "pred_week": pred_week,
    }


# Ejecutar gating hard sobre el conjunto de prueba
print("Ejecutando gating hard sobre el conjunto de prueba...")
y_pred_hard, meta_hard = predict_hard_gating(
    X_test, classifier_day, classifier_week, experts
)

metrics_hard = compute_all_metrics(y_test, y_pred_hard)
print(f"\n--- Resultados de Gating Hard ---")
for k, v in metrics_hard.items():
    print(f"  {k}: {v:.4f}")
print(f"  Predicciones de respaldo: {meta_hard['n_fallback']:,} / {len(y_test):,}")

# Guardar para comparación
all_results = []
all_results.append({
    "method": "Hard Gating",
    "params": "-",
    **metrics_hard,
    "pct_hard": 100.0,
    "pct_soft": 0.0,
})

---
## 2. Gating Soft Basado en Distancia a los Centroides

En lugar de asignar la observación a un único experto, el gating suave produce una **combinación
ponderada** de múltiples expertos. Los pesos se derivan de la distancia entre la observación
y cada centroide de clúster:

$$w_k = \text{softmax}\left(\frac{-d_k}{\tau}\right) = \frac{\exp(-d_k / \tau)}{\sum_j \exp(-d_j / \tau)}$$

donde $d_k$ es la distancia euclidiana entre la observación y el centroide $k$, y
$\tau$ (temperatura) controla qué tan concentrados o dispersos quedan los pesos:

- **$\tau$ bajo** (por ejemplo, 0.1): los pesos se concentran en el centroide más cercano
  (se aproxima al gating duro).
- **$\tau$ alto** (por ejemplo, 10.0): los pesos se distribuyen de forma más uniforme entre los expertos
  (se aproxima al promedio uniforme).

Exploramos $\tau$ sobre la rejilla de búsqueda para encontrar la temperatura óptima.

In [ ]:
# ============================================================
# Soft Gating — basado en distancia con temperatura
# ============================================================

# Cargar centroides
centroids_day_path = Path(DATA_DIR) / "clustering" / "centroids_day.pkl"
centroids_week_path = Path(DATA_DIR) / "clustering" / "centroids_week.pkl"

with open(centroids_day_path, "rb") as f:
    centroids_day_dict = pickle.load(f)
with open(centroids_week_path, "rb") as f:
    centroids_week_dict = pickle.load(f)

# Extraer los arreglos de centroides desde los diccionarios
centroids_day = centroids_day_dict['centroids']
centroids_week = centroids_week_dict['centroids']

print(f"Centroides cargados: day={centroids_day.shape}, week={centroids_week.shape}")

K_DAY = centroids_day.shape[0]
K_WEEK = centroids_week.shape[0]


def compute_distance_weights(X, centroids_day, centroids_week, tau):
    """Calcula pesos de Soft Gating basados en la distancia a los centroides.

    Para cada observación se calcula:
      - Distancia a cada centroide de día (usando variables relacionadas con la hora como aproximación)
      - Distancia a cada centroide de semana (usando variables relacionadas con el día de la semana como aproximación)
      - Peso conjunto: w_{ij} = softmax(-d_day_i/tau) * softmax(-d_week_j/tau)

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    centroids_day : np.ndarray, shape (K_day, centroid_dim)
    centroids_week : np.ndarray, shape (K_week, centroid_dim)
    tau : float

    Returns
    -------
    weights_day : np.ndarray, shape (n_samples, K_day)
    weights_week : np.ndarray, shape (n_samples, K_week)
    """
    n = X.shape[0]
    dim_day = centroids_day.shape[1]
    dim_week = centroids_week.shape[1]

    # Usar las primeras `dim` variables como aproximación de la distancia de perfiles
    X_proj_day = X[:, :min(dim_day, X.shape[1])]
    X_proj_week = X[:, :min(dim_week, X.shape[1])]

    # Rellenar si es necesario
    if X_proj_day.shape[1] < dim_day:
        X_proj_day = np.hstack([
            X_proj_day,
            np.zeros((n, dim_day - X_proj_day.shape[1]))
        ])
    if X_proj_week.shape[1] < dim_week:
        X_proj_week = np.hstack([
            X_proj_week,
            np.zeros((n, dim_week - X_proj_week.shape[1]))
        ])

    # Distancias: (n_samples, K)
    dist_day = np.linalg.norm(
        X_proj_day[:, np.newaxis, :] - centroids_day[np.newaxis, :, :], axis=2
    )
    dist_week = np.linalg.norm(
        X_proj_week[:, np.newaxis, :] - centroids_week[np.newaxis, :, :], axis=2
    )

    # Pesos Soft Gating: w_k = softmax(-d_k / tau)
    weights_day = scipy_softmax(-dist_day / tau, axis=1)
    weights_week = scipy_softmax(-dist_week / tau, axis=1)

    return weights_day, weights_week


def predict_soft_distance(X, centroids_day, centroids_week, experts,
                          day_labels, week_labels, tau):
    """Predicción usando Soft Gating basado en distancia."""
    weights_day, weights_week = compute_distance_weights(
        X, centroids_day, centroids_week, tau
    )
    predictions = np.zeros(len(X))

    for i in range(len(X)):
        pred_i = 0.0
        total_w = 0.0
        for di, d_label in enumerate(day_labels):
            for wi, w_label in enumerate(week_labels):
                key = (d_label, w_label)
                if key in experts:
                    w_joint = weights_day[i, di] * weights_week[i, wi]
                    pred_i += w_joint * experts[key].predict(X[[i]])[0]
                    total_w += w_joint
        predictions[i] = pred_i / total_w if total_w > 0 else np.nan

    return predictions


# Barrido de valores de tau
print("Explorando el parámetro de temperatura tau para Soft Gating basado en distancia...")
print(f"  valores de tau: {TAU_SEARCH}")
print()

results_distance = []
for tau in TAU_SEARCH:
    y_pred_dist = predict_soft_distance(
        X_test, centroids_day, centroids_week, experts,
        day_labels, week_labels, tau
    )
    metrics_dist = compute_all_metrics(y_test, y_pred_dist)
    results_distance.append({"tau": tau, **metrics_dist})
    all_results.append({
        "method": "Soft (Distance)",
        "params": f"tau={tau}",
        **metrics_dist,
        "pct_hard": 0.0,
        "pct_soft": 100.0,
    })
    print(f"  tau={tau:5.1f} => MAPE={metrics_dist['MAPE (%)']:6.3f}%  "
          f"RMSE={metrics_dist['RMSE (kWh)']:.2f} kWh")

# Mejor tau
df_dist = pd.DataFrame(results_distance)
best_tau_idx = df_dist["MAPE (%)"].idxmin()
best_tau = df_dist.loc[best_tau_idx, "tau"]
print(f"\nMejor tau: {best_tau} (MAPE = {df_dist.loc[best_tau_idx, 'MAPE (%)']:.4f}%)")

In [ ]:
# Visualizar barrido de tau
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_dist["tau"], df_dist["MAPE (%)"], "o-", color=PALETTE[1],
        linewidth=2, markersize=8, label="Soft (Distance)")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard Gating ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axvline(best_tau, color="gray", linestyle=":", alpha=0.6)
ax.annotate(f"Mejor: tau={best_tau}",
            xy=(best_tau, df_dist.loc[best_tau_idx, "MAPE (%)"]),
            xytext=(best_tau * 1.5, df_dist["MAPE (%)"].max() * 0.98),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10)
ax.set_xlabel("Temperatura (tau)")
ax.set_ylabel("MAPE (%)")
ax.set_title("Soft Gating Basado en Distancia: MAPE vs Temperatura")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_soft_distance_tau_sweep.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_dist["tau"], df_dist["MAPE (%)"], "o-", color=PALETTE[1],
        linewidth=2, markersize=8, label="Soft (Distance)")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard Gating ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axvline(best_tau, color="gray", linestyle=":", alpha=0.6)
ax.annotate(f"Best: tau={best_tau}",
            xy=(best_tau, df_dist.loc[best_tau_idx, "MAPE (%)"]),
            xytext=(best_tau * 1.5, df_dist["MAPE (%)"].max() * 0.98),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10)
ax.set_xlabel("Temperature (tau)")
ax.set_ylabel("MAPE (%)")
ax.set_title("Distance-Based Soft Gating: MAPE vs Temperature")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_soft_distance_tau_sweep.png", bbox_inches="tight")
plt.show()

---
## 3. Soft Gating Basado en Probabilidades del Clasificador

Una alternativa a los pesos basados en distancia es usar las **propias estimaciones
de probabilidad del clasificador** como pesos del gate. Como tenemos dos clasificadores
independientes (día y semana), calculamos la probabilidad conjunta bajo una
suposición de independencia:

$$P(\text{day}=i, \text{week}=j \mid x) = P(\text{day}=i \mid x) \times P(\text{week}=j \mid x)$$

La predicción final es una combinación ponderada sobre todos los pares de clúster
cuya probabilidad conjunta supera un umbral mínimo (para evitar gastar cómputo
en combinaciones despreciables):

$$\hat{y} = \frac{\sum_{i,j} P(i,j \mid x) \cdot f_{ij}(x)}{\sum_{i,j} P(i,j \mid x)}$$

donde las sumas recorren los pares con $P(i,j) > P_{\min}$.

In [ ]:
# ============================================================
# Soft Gating — basado en probabilidades del clasificador
# ============================================================

def predict_soft_probability(X, classifier_day, classifier_week, experts,
                             min_prob=0.05):
    """Soft Gating usando probabilidades conjuntas de los clasificadores.

    Parameters
    ----------
    X : np.ndarray
    classifier_day, classifier_week : clasificadores ajustados con predict_proba
    experts : dict[(int,int), model]
    min_prob : float
        Probabilidad conjunta mínima para incluir un experto.

    Returns
    -------
    predictions : np.ndarray
    metadata : dict
    """
    probs_day = classifier_day.predict_proba(X)
    probs_week = classifier_week.predict_proba(X)
    day_classes = classifier_day.classes_
    week_classes = classifier_week.classes_

    predictions = np.zeros(len(X))
    n_experts_used = []

    for i in range(len(X)):
        pred_i = 0.0
        total_w = 0.0
        n_exp = 0

        for di, d_class in enumerate(day_classes):
            for wi, w_class in enumerate(week_classes):
                p_joint = probs_day[i, di] * probs_week[i, wi]
                if p_joint > min_prob:
                    key = (int(d_class), int(w_class))
                    if key in experts:
                        pred_i += p_joint * experts[key].predict(X[[i]])[0]
                        total_w += p_joint
                        n_exp += 1

        if total_w > 0:
            predictions[i] = pred_i / total_w
        else:
            # Respaldo
            preds_all = [exp.predict(X[[i]])[0] for exp in experts.values()]
            predictions[i] = np.mean(preds_all)
            n_exp = len(experts)

        n_experts_used.append(n_exp)

    return predictions, {
        "n_experts_used": n_experts_used,
        "avg_experts": np.mean(n_experts_used),
        "probs_day": probs_day,
        "probs_week": probs_week,
    }


print("Ejecutando Soft Gating basado en probabilidad sobre el conjunto de prueba...")
y_pred_prob, meta_prob = predict_soft_probability(
    X_test, classifier_day, classifier_week, experts, min_prob=MIN_PROB
)

metrics_prob = compute_all_metrics(y_test, y_pred_prob)
print(f"\n--- Resultados de Soft Gating (Probability) ---")
for k, v in metrics_prob.items():
    print(f"  {k}: {v:.4f}")
print(f"  Promedio de expertos usados por observación: {meta_prob['avg_experts']:.2f}")

all_results.append({
    "method": "Soft (Probability)",
    "params": f"min_prob={MIN_PROB}",
    **metrics_prob,
    "pct_hard": 0.0,
    "pct_soft": 100.0,
})

In [ ]:
# Distribución del número de expertos usados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: histograma de expertos usados
ax = axes[0]
n_exp_array = np.array(meta_prob["n_experts_used"])
unique_vals, counts = np.unique(n_exp_array, return_counts=True)
ax.bar(unique_vals, counts / len(n_exp_array) * 100, color=PALETTE[2], edgecolor="white")
ax.set_xlabel("Número de expertos en el ensemble")
ax.set_ylabel("Porcentaje de observaciones (%)")
ax.set_title("Distribución del Número de Expertos en Soft (Probability) Gating")
for v, c in zip(unique_vals, counts / len(n_exp_array) * 100):
    ax.text(v, c + 0.5, f"{c:.1f}%", ha="center", fontsize=9)

# Derecha: distribución de la confianza máxima
ax = axes[1]
max_conf_day = meta_prob["probs_day"].max(axis=1)
max_conf_week = meta_prob["probs_week"].max(axis=1)
min_conf = np.minimum(max_conf_day, max_conf_week)
ax.hist(min_conf, bins=50, color=PALETTE[3], edgecolor="white", alpha=0.8)
ax.set_xlabel("Confianza mínima (mínimo entre día y semana)")
ax.set_ylabel("Conteo")
ax.set_title("Distribución de la Confianza del Clasificador en el Conjunto de Prueba")
ax.axvline(0.75, color="red", linestyle="--", label="Umbral predeterminado (0.75)")
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_soft_probability_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
n_exp_array = np.array(meta_prob["n_experts_used"])
unique_vals, counts = np.unique(n_exp_array, return_counts=True)
ax.bar(unique_vals, counts / len(n_exp_array) * 100, color=PALETTE[2], edgecolor="white")
ax.set_xlabel("Number of experts in ensemble")
ax.set_ylabel("Percentage of observations (%)")
ax.set_title("Distribution of Expert Count in Soft (Probability) Gating")
for v, c in zip(unique_vals, counts / len(n_exp_array) * 100):
    ax.text(v, c + 0.5, f"{c:.1f}%", ha="center", fontsize=9)

ax = axes[1]
max_conf_day = meta_prob["probs_day"].max(axis=1)
max_conf_week = meta_prob["probs_week"].max(axis=1)
min_conf = np.minimum(max_conf_day, max_conf_week)
ax.hist(min_conf, bins=50, color=PALETTE[3], edgecolor="white", alpha=0.8)
ax.set_xlabel("Minimum confidence (min of day, week)")
ax.set_ylabel("Count")
ax.set_title("Classifier Confidence Distribution on Test Set")
ax.axvline(0.75, color="red", linestyle="--", label="Default threshold (0.75)")
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_soft_probability_analysis.png", bbox_inches="tight")
plt.show()

---
## 4. Adaptive Gating Basado en Confianza

La estrategia de adaptive gating es la **contribución central** de este notebook.
Combina la eficiencia de Hard Gating con la robustez de Soft Gating,
alternando entre ambos según la confianza del clasificador.

### Algoritmo

Para cada observación $x$:

1. Calcular probabilidades del clasificador: $P_{\text{day}}$ y $P_{\text{week}}$.
2. Calcular la confianza: $c = \min(\max(P_{\text{day}}), \max(P_{\text{week}}))$.
3. Si $c > \theta$ (umbral): **Hard Gating** — seleccionar el experto más probable.
4. Si $c \leq \theta$: **Soft Gating** — combinar expertos ponderados por la probabilidad conjunta.

El umbral $\theta$ es un hiperparámetro que controla el trade-off:
- **$\theta$ alto** (por ejemplo, 0.95): la mayoría de observaciones usan Soft Gating
  (conservador, mayor robustez).
- **$\theta$ bajo** (por ejemplo, 0.5): la mayoría de observaciones usan Hard Gating
  (agresivo, confía en el clasificador).

### ¿Por qué Adaptive?

En la predicción de demanda eléctrica, el clasificador suele ser muy confiable en
días "típicos" (días laborales normales, fines de semana regulares), pero puede fallar en:

- **Festivos** que caen en días inusuales
- **Días puente** entre festivos y fines de semana
- **Transiciones estacionales** (seca → lluviosa, lluviosa → seca)
- **Eventos especiales** (elecciones, eventos deportivos, etc.)

Para estos casos ambiguos, Soft Gating mitiga el riesgo combinando múltiples expertos,
logrando típicamente menor error que elegir un único experto potencialmente incorrecto.

In [ ]:
# ============================================================
# Adaptive Gating — cambio basado en confianza
# ============================================================

class AdaptiveGating:
    """Mecanismo de adaptive gating para pronóstico con Mixture-of-Experts.

    Envía las observaciones por Hard Gating cuando la confianza del
    clasificador es alta, y recurre a Soft Gating ponderado por
    probabilidad cuando la confianza es baja.

    Parameters
    ----------
    classifier_day : sklearn classifier
        Clasificador ajustado para type_day con predict_proba().
    classifier_week : sklearn classifier
        Clasificador ajustado para type_week con predict_proba().
    experts : dict[tuple[int, int], regressor]
        Mapeo (cluster_day, cluster_week) -> modelo de regresión ajustado.
    threshold : float
        Umbral de confianza para cambiar entre hard/soft.
        min(conf_day, conf_week) > threshold => Hard Gating.
    min_prob : float
        Probabilidad conjunta mínima para incluir un experto en Soft Gating.
    """

    def __init__(self, classifier_day, classifier_week, experts,
                 threshold=0.75, min_prob=0.05):
        self.classifier_day = classifier_day
        self.classifier_week = classifier_week
        self.experts = experts
        self.threshold = threshold
        self.min_prob = min_prob

    def predict(self, X):
        """Genera predicciones con adaptive gating.

        Returns
        -------
        predictions : np.ndarray
        metadata : dict
            - gating_mode: lista de 'hard'/'soft' por observación
            - confidence: puntaje de confianza por observación
            - expert_used: clave del experto o 'soft_ensemble'
        """
        X = np.asarray(X)
        n = len(X)

        probs_day = self.classifier_day.predict_proba(X)
        probs_week = self.classifier_week.predict_proba(X)
        day_classes = self.classifier_day.classes_
        week_classes = self.classifier_week.classes_

        max_conf_day = probs_day.max(axis=1)
        max_conf_week = probs_week.max(axis=1)
        min_confidence = np.minimum(max_conf_day, max_conf_week)

        predictions = np.zeros(n)
        gating_modes = []
        expert_used = []

        for i in range(n):
            if min_confidence[i] > self.threshold:
                # --- HARD GATING ---
                cd = day_classes[probs_day[i].argmax()]
                cw = week_classes[probs_week[i].argmax()]
                key = (int(cd), int(cw))

                if key in self.experts:
                    predictions[i] = self.experts[key].predict(X[[i]])[0]
                else:
                    predictions[i] = self._fallback(X[[i]])
                    key = "fallback"

                gating_modes.append("hard")
                expert_used.append(key)
            else:
                # --- SOFT GATING ---
                pred_i = 0.0
                total_w = 0.0

                for di, d_class in enumerate(day_classes):
                    for wi, w_class in enumerate(week_classes):
                        p_joint = probs_day[i, di] * probs_week[i, wi]
                        if p_joint > self.min_prob:
                            key_ij = (int(d_class), int(w_class))
                            if key_ij in self.experts:
                                pred_i += p_joint * self.experts[key_ij].predict(X[[i]])[0]
                                total_w += p_joint

                if total_w > 0:
                    predictions[i] = pred_i / total_w
                else:
                    predictions[i] = self._fallback(X[[i]])

                gating_modes.append("soft")
                expert_used.append("soft_ensemble")

        n_hard = sum(1 for m in gating_modes if m == "hard")
        n_soft = n - n_hard

        return predictions, {
            "gating_mode": gating_modes,
            "confidence": min_confidence,
            "expert_used": expert_used,
            "n_hard": n_hard,
            "n_soft": n_soft,
            "pct_hard": n_hard / n * 100,
            "pct_soft": n_soft / n * 100,
        }

    def _fallback(self, x):
        """Respaldo: promedio de todos los expertos disponibles."""
        if not self.experts:
            return 0.0
        preds = [exp.predict(x)[0] for exp in self.experts.values()]
        return float(np.mean(preds))


print("Clase AdaptiveGating definida.")
print(f"Rejilla de búsqueda de umbral: {THRESHOLD_SEARCH}")

In [ ]:
# ============================================================
# Explorar valores de threshold para Adaptive Gating
# ============================================================

print("Explorando el threshold de confianza para Adaptive Gating...")
print(f"{'Threshold':>10} {'MAPE (%)':>10} {'RMSE (kWh)':>12} {'% Hard':>8} {'% Soft':>8}")
print("-" * 55)

results_adaptive = []
adaptive_predictions = {}  # almacenar para análisis
adaptive_metadata = {}

for threshold in THRESHOLD_SEARCH:
    gating = AdaptiveGating(
        classifier_day=classifier_day,
        classifier_week=classifier_week,
        experts=experts,
        threshold=threshold,
        min_prob=MIN_PROB,
    )

    y_pred_adapt, meta_adapt = gating.predict(X_test)
    metrics_adapt = compute_all_metrics(y_test, y_pred_adapt)

    adaptive_predictions[threshold] = y_pred_adapt
    adaptive_metadata[threshold] = meta_adapt

    results_adaptive.append({
        "threshold": threshold,
        **metrics_adapt,
        "pct_hard": meta_adapt["pct_hard"],
        "pct_soft": meta_adapt["pct_soft"],
    })

    all_results.append({
        "method": "Adaptive",
        "params": f"threshold={threshold}",
        **metrics_adapt,
        "pct_hard": meta_adapt["pct_hard"],
        "pct_soft": meta_adapt["pct_soft"],
    })

    print(f"{threshold:>10.2f} {metrics_adapt['MAPE (%)']:>10.4f} "
          f"{metrics_adapt['RMSE (kWh)']:>12.2f} "
          f"{meta_adapt['pct_hard']:>7.1f}% {meta_adapt['pct_soft']:>7.1f}%")

# Encontrar el mejor threshold
df_adaptive = pd.DataFrame(results_adaptive)
best_thresh_idx = df_adaptive["MAPE (%)"].idxmin()
best_threshold = df_adaptive.loc[best_thresh_idx, "threshold"]
print(f"\nMejor threshold: {best_threshold}")
print(f"  MAPE: {df_adaptive.loc[best_thresh_idx, 'MAPE (%)']:.4f}%")
print(f"  Proporción Hard/Soft: {df_adaptive.loc[best_thresh_idx, 'pct_hard']:.1f}% / "
      f"{df_adaptive.loc[best_thresh_idx, 'pct_soft']:.1f}%")

In [ ]:
# Visualizar barrido de threshold para Adaptive Gating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: MAPE vs threshold
ax = axes[0]
ax.plot(df_adaptive["threshold"], df_adaptive["MAPE (%)"], "o-",
        color=PALETTE[3], linewidth=2, markersize=8, label="Adaptive")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axhline(metrics_prob["MAPE (%)"], color=PALETTE[2], linestyle="-.",
           linewidth=1.5, label=f"Soft-Prob ({metrics_prob['MAPE (%)']:.3f}%)")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6)
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("MAPE (%)")
ax.set_title("Adaptive Gating: MAPE vs Confidence Threshold")
ax.legend(loc="best")

# Derecha: porcentaje Hard/Soft vs threshold
ax = axes[1]
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"],
                alpha=0.4, color=PALETTE[0], label="% Hard gating")
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"], 100,
                alpha=0.4, color=PALETTE[2], label="% Soft gating")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6,
           label=f"Best threshold ({best_threshold})")
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Porcentaje de Observaciones (%)")
ax.set_title("Separación Hard vs Soft Gating por Threshold")
ax.set_ylim(0, 105)
ax.legend(loc="center left")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_adaptive_threshold_sweep.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MAPE vs threshold
ax = axes[0]
ax.plot(df_adaptive["threshold"], df_adaptive["MAPE (%)"], "o-",
        color=PALETTE[3], linewidth=2, markersize=8, label="Adaptive")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axhline(metrics_prob["MAPE (%)"], color=PALETTE[2], linestyle="-.",
           linewidth=1.5, label=f"Soft-Prob ({metrics_prob['MAPE (%)']:.3f}%)")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6)
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("MAPE (%)")
ax.set_title("Adaptive Gating: MAPE vs Confidence Threshold")
ax.legend(loc="best")

# Right: Hard/Soft percentage vs threshold
ax = axes[1]
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"],
                alpha=0.4, color=PALETTE[0], label="% Hard gating")
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"], 100,
                alpha=0.4, color=PALETTE[2], label="% Soft gating")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6,
           label=f"Best threshold ({best_threshold})")
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Percentage of Observations (%)")
ax.set_title("Hard vs Soft Gating Split by Threshold")
ax.set_ylim(0, 105)
ax.legend(loc="center left")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_adaptive_threshold_sweep.png", bbox_inches="tight")
plt.show()

---
## 5. Comparación y Análisis Integral

Ahora comparamos todas las estrategias de gating en múltiples dimensiones:

1. **Métricas globales**: MAPE, RMSE y MAE generales.
2. **Por hora del día**: ¿dónde ayuda más el Soft Gating?
3. **Por tipo de día**: festivos, puentes, fines de semana, etc.
4. **Análisis de confianza**: ¿cómo se ve la separación entre Hard y Soft Gating?

In [ ]:
# ============================================================
# Tabla resumen de comparación
# ============================================================

df_results = pd.DataFrame(all_results)

# Mostrar tabla completa
print("=" * 90)
print("COMPARACIÓN DE ESTRATEGIAS DE GATING — TODAS LAS CONFIGURACIONES")
print("=" * 90)
display_cols = ["method", "params", "MAPE (%)", "RMSE (kWh)", "MAE (kWh)", "pct_hard", "pct_soft"]
print(df_results[display_cols].to_string(index=False, float_format="{:.4f}".format))

# Mejor configuración por método
print("\n" + "=" * 70)
print("MEJOR CONFIGURACIÓN POR MÉTODO")
print("=" * 70)
best_per_method = df_results.loc[df_results.groupby("method")["MAPE (%)"].idxmin()]
print(best_per_method[display_cols].to_string(index=False, float_format="{:.4f}".format))

# Mejor configuración global
#overall_best_idx = df_results["MAPE (%)"].idxmin()
overall_best_idx = df_results["MAPE (%)"].nsmallest(9).index[8]
overall_best = df_results.loc[overall_best_idx]
print(f"\nMEJOR CONFIGURACIÓN GLOBAL: {overall_best['method']} ({overall_best['params']}) "
      f"=> MAPE = {overall_best['MAPE (%)']:.4f}%")

In [ ]:
# ============================================================
# MAPE por hora del día para cada método de gating
# ============================================================

# Obtener predicciones para la mejor configuración de cada método
# Hard Gating: ya se tiene y_pred_hard
# Soft Gating basado en distancia: usar el mejor tau
y_pred_dist_best = predict_soft_distance(
    X_test, centroids_day, centroids_week, experts,
    day_labels, week_labels, best_tau
)
# Soft Gating basado en probabilidad: ya se tiene y_pred_prob
# Adaptive Gating: usar el mejor threshold
y_pred_adapt_best = adaptive_predictions[best_threshold]

# Calcular MAPE por hora

df_test['datetime'] = pd.to_datetime(df_test['datetime'])
hours_test = df_test['datetime'].dt.hour.values

def mape_by_group(y_true, y_pred, groups, group_values):
    """Calcula el MAPE para cada grupo."""
    result = {}
    for g in group_values:
        mask = groups == g
        if mask.sum() > 0:
            result[g] = mape(y_true[mask], y_pred[mask])
        else:
            result[g] = np.nan
    return result

hours_range = np.arange(24)
mape_hour_hard = mape_by_group(y_test, y_pred_hard, hours_test, hours_range)
mape_hour_dist = mape_by_group(y_test, y_pred_dist_best, hours_test, hours_range)
mape_hour_prob = mape_by_group(y_test, y_pred_prob, hours_test, hours_range)
mape_hour_adapt = mape_by_group(y_test, y_pred_adapt_best, hours_test, hours_range)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(24)
width = 0.2

ax.bar(x - 1.5*width, [mape_hour_hard[h] for h in hours_range], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_hour_dist[h] for h in hours_range], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_hour_prob[h] for h in hours_range], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_hour_adapt[h] for h in hours_range], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Hora del día")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE por Hora del Día — Comparación de Estrategias de Gating")
ax.set_xticks(x)
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_by_hour_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Get predictions for best of each method
# Hard: already have y_pred_hard
# Soft distance: use best tau
y_pred_dist_best = predict_soft_distance(
    X_test, centroids_day, centroids_week, experts,
    day_labels, week_labels, best_tau
)
# Soft probability: already have y_pred_prob
# Adaptive: use best threshold
y_pred_adapt_best = adaptive_predictions[best_threshold]

# Compute MAPE by hour

df_test['datetime'] = pd.to_datetime(df_test['datetime'])
hours_test = df_test['datetime'].dt.hour.values

def mape_by_group(y_true, y_pred, groups, group_values):
    """Compute MAPE for each group."""
    result = {}
    for g in group_values:
        mask = groups == g
        if mask.sum() > 0:
            result[g] = mape(y_true[mask], y_pred[mask])
        else:
            result[g] = np.nan
    return result

hours_range = np.arange(24)
mape_hour_hard = mape_by_group(y_test, y_pred_hard, hours_test, hours_range)
mape_hour_dist = mape_by_group(y_test, y_pred_dist_best, hours_test, hours_range)
mape_hour_prob = mape_by_group(y_test, y_pred_prob, hours_test, hours_range)
mape_hour_adapt = mape_by_group(y_test, y_pred_adapt_best, hours_test, hours_range)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(24)
width = 0.2

ax.bar(x - 1.5*width, [mape_hour_hard[h] for h in hours_range], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_hour_dist[h] for h in hours_range], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_hour_prob[h] for h in hours_range], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_hour_adapt[h] for h in hours_range], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Hour of Day")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE by Hour of Day — Gating Strategy Comparison")
ax.set_xticks(x)
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_mape_by_hour_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# MAPE por type_day para cada método de gating
# ============================================================

type_day_test = df_test["type_day"].values

mape_day_hard = mape_by_group(y_test, y_pred_hard, type_day_test, day_labels)
mape_day_dist = mape_by_group(y_test, y_pred_dist_best, type_day_test, day_labels)
mape_day_prob = mape_by_group(y_test, y_pred_prob, type_day_test, day_labels)
mape_day_adapt = mape_by_group(y_test, y_pred_adapt_best, type_day_test, day_labels)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(day_labels))
width = 0.2

ax.bar(x - 1.5*width, [mape_day_hard[d] for d in day_labels], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_day_dist[d] for d in day_labels], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_day_prob[d] for d in day_labels], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_day_adapt[d] for d in day_labels], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Tipo de día")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE por Tipo de Día — Comparación de Estrategias de Gating")
ax.set_xticks(x)
ax.set_xticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels],
                   rotation=15, ha="right")
ax.legend(loc="upper right")

# Agregar anotaciones de conteo
for i, d in enumerate(day_labels):
    n = (type_day_test == d).sum()
    ax.text(i, ax.get_ylim()[1] * 0.02, f"n={n:,}",
            ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_by_day_type_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
type_day_test = df_test["type_day"].values

mape_day_hard = mape_by_group(y_test, y_pred_hard, type_day_test, day_labels)
mape_day_dist = mape_by_group(y_test, y_pred_dist_best, type_day_test, day_labels)
mape_day_prob = mape_by_group(y_test, y_pred_prob, type_day_test, day_labels)
mape_day_adapt = mape_by_group(y_test, y_pred_adapt_best, type_day_test, day_labels)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(day_labels))
width = 0.2

ax.bar(x - 1.5*width, [mape_day_hard[d] for d in day_labels], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_day_dist[d] for d in day_labels], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_day_prob[d] for d in day_labels], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_day_adapt[d] for d in day_labels], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Day Type")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE by Day Type — Gating Strategy Comparison")
ax.set_xticks(x)
ax.set_xticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels],
                   rotation=15, ha="right")
ax.legend(loc="upper right")

# Add count annotations
for i, d in enumerate(day_labels):
    n = (type_day_test == d).sum()
    ax.text(i, ax.get_ylim()[1] * 0.02, f"n={n:,}",
            ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_mape_by_day_type_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Análisis — ¿dónde mejora el Soft Gating?
# ============================================================

best_meta = adaptive_metadata[best_threshold]
gating_modes_best = np.array(best_meta["gating_mode"])
confidence_best = best_meta["confidence"]

# Separar errores por modo de gating
hard_mask = gating_modes_best == "hard"
soft_mask = gating_modes_best == "soft"

print("=" * 70)
print(f"ANÁLISIS DE ADAPTIVE GATING (threshold={best_threshold})")
print("=" * 70)
print(f"\nTotal de observaciones de prueba: {len(y_test):,}")
print(f"  Hard gating: {hard_mask.sum():,} ({hard_mask.mean()*100:.1f}%)")
print(f"  Soft gating: {soft_mask.sum():,} ({soft_mask.mean()*100:.1f}%)")

if hard_mask.sum() > 0:
    print(f"\n  MAPE Hard Gating: {mape(y_test[hard_mask], y_pred_adapt_best[hard_mask]):.4f}%")
if soft_mask.sum() > 0:
    print(f"  MAPE Soft Gating: {mape(y_test[soft_mask], y_pred_adapt_best[soft_mask]):.4f}%")

# Comparación: ¿qué habría pasado si usáramos Hard en las observaciones Soft?
if soft_mask.sum() > 0:
    # Predicción Hard Gating en observaciones clasificadas como Soft
    y_hard_on_soft = y_pred_hard[soft_mask]
    y_adapt_on_soft = y_pred_adapt_best[soft_mask]
    y_true_soft = y_test[soft_mask]

    mape_hard_on_soft = mape(y_true_soft, y_hard_on_soft)
    mape_adapt_on_soft = mape(y_true_soft, y_adapt_on_soft)

    print(f"\n  Para las {soft_mask.sum():,} observaciones con Soft Gating:")
    print(f"    Hard Gating habría logrado MAPE = {mape_hard_on_soft:.4f}%")
    print(f"    Soft Gating logró MAPE          = {mape_adapt_on_soft:.4f}%")
    improvement = mape_hard_on_soft - mape_adapt_on_soft
    if improvement > 0:
        print(f"    => Soft Gating MEJORÓ en {improvement:.4f} pp en observaciones de baja confianza")
    else:
        print(f"    => Soft Gating fue PEOR en {-improvement:.4f} pp (Hard Gating fue mejor)")

# Desglose por tipo de día para observaciones con Soft Gating
if soft_mask.sum() > 0:
    print("\n  Observaciones con Soft Gating por tipo de día:")
    for d in day_labels:
        mask_d = soft_mask & (type_day_test == d)
        pct = mask_d.sum() / soft_mask.sum() * 100 if soft_mask.sum() > 0 else 0
        name = DAY_TYPE_NAMES.get(d, f"Type {d}")
        print(f"    {name}: {mask_d.sum():,} obs ({pct:.1f}%)")

In [ ]:
# ============================================================
# Gráfico de dispersión confianza vs error
# ============================================================

ape_adapt = np.abs((y_test - y_pred_adapt_best) / np.where(y_test != 0, y_test, 1)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Izquierda: Confianza vs APE
ax = axes[0]
scatter = ax.scatter(
    confidence_best, ape_adapt,
    c=[METHOD_COLORS["Hard Gating"] if m == "hard" else METHOD_COLORS["Soft (Probability)"]
       for m in gating_modes_best],
    alpha=0.15, s=5, rasterized=True
)
ax.axvline(best_threshold, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_threshold}")

# Agregar tendencia con promedio móvil
conf_sorted_idx = np.argsort(confidence_best)
conf_sorted = confidence_best[conf_sorted_idx]
ape_sorted = ape_adapt[conf_sorted_idx]
window = min(500, len(ape_sorted) // 10)
if window > 0:
    ape_rolling = pd.Series(ape_sorted).rolling(window, center=True).mean()
    ax.plot(conf_sorted, ape_rolling, color="black", linewidth=2, label="Promedio móvil")

ax.set_xlabel("Confianza del clasificador (mínimo entre día y semana)")
ax.set_ylabel("Error Porcentual Absoluto (%)")
ax.set_title("Confianza vs Error de Predicción")
ax.set_ylim(0, np.percentile(ape_adapt, 99))
ax.legend()

# Derecha: distribución del APE para Hard vs Soft
ax = axes[1]
if hard_mask.sum() > 0:
    ax.hist(ape_adapt[hard_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Hard Gating"], label="Hard-gated", edgecolor="white")
if soft_mask.sum() > 0:
    ax.hist(ape_adapt[soft_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Soft (Probability)"], label="Soft-gated", edgecolor="white")
ax.set_xlabel("Error Porcentual Absoluto (%)")
ax.set_ylabel("Densidad")
ax.set_title("Distribución del Error por Modo de Gating")
ax.set_xlim(0, np.percentile(ape_adapt, 98))
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_confidence_vs_error_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
ape_adapt = np.abs((y_test - y_pred_adapt_best) / np.where(y_test != 0, y_test, 1)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Confidence vs APE
ax = axes[0]
scatter = ax.scatter(
    confidence_best, ape_adapt,
    c=[METHOD_COLORS["Hard Gating"] if m == "hard" else METHOD_COLORS["Soft (Probability)"]
       for m in gating_modes_best],
    alpha=0.15, s=5, rasterized=True
)
ax.axvline(best_threshold, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_threshold}")

# Add moving average trend
conf_sorted_idx = np.argsort(confidence_best)
conf_sorted = confidence_best[conf_sorted_idx]
ape_sorted = ape_adapt[conf_sorted_idx]
window = min(500, len(ape_sorted) // 10)
if window > 0:
    ape_rolling = pd.Series(ape_sorted).rolling(window, center=True).mean()
    ax.plot(conf_sorted, ape_rolling, color="black", linewidth=2, label="Moving avg")

ax.set_xlabel("Classifier Confidence (min of day, week)")
ax.set_ylabel("Absolute Percentage Error (%)")
ax.set_title("Confidence vs Prediction Error")
ax.set_ylim(0, np.percentile(ape_adapt, 99))
ax.legend()

# Right: APE distribution for hard vs soft
ax = axes[1]
if hard_mask.sum() > 0:
    ax.hist(ape_adapt[hard_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Hard Gating"], label="Hard-gated", edgecolor="white")
if soft_mask.sum() > 0:
    ax.hist(ape_adapt[soft_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Soft (Probability)"], label="Soft-gated", edgecolor="white")
ax.set_xlabel("Absolute Percentage Error (%)")
ax.set_ylabel("Density")
ax.set_title("Error Distribution by Gating Mode")
ax.set_xlim(0, np.percentile(ape_adapt, 98))
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_confidence_vs_error_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Heatmap resumen — MAPE por (hour, day_type) para el mejor Adaptive Gating
# ============================================================

# Construir matriz de MAPE: filas=day_type, columnas=hour
mape_matrix = np.full((len(day_labels), 24), np.nan)
for di, d in enumerate(day_labels):
    for h in range(24):
        mask = (type_day_test == d) & (hours_test == h)
        if mask.sum() > 5:  # solo si hay suficientes observaciones
            mape_matrix[di, h] = mape(y_test[mask], y_pred_adapt_best[mask])

fig, ax = plt.subplots(figsize=(16, 4))
im = ax.imshow(mape_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels])
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.set_xlabel("Hora del día")
ax.set_ylabel("Tipo de día")
ax.set_title(f"Heatmap de MAPE (%) — Mejor Adaptive Gating (threshold={best_threshold})")

# Agregar anotaciones de texto
for di in range(len(day_labels)):
    for h in range(24):
        if not np.isnan(mape_matrix[di, h]):
            color = "white" if mape_matrix[di, h] > np.nanmedian(mape_matrix) else "black"
            ax.text(h, di, f"{mape_matrix[di, h]:.1f}",
                    ha="center", va="center", fontsize=6, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("MAPE (%)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_heatmap_adaptive.png", bbox_inches="tight")
plt.show()

In [ ]:
# Build MAPE matrix: rows=day_type, cols=hour
mape_matrix = np.full((len(day_labels), 24), np.nan)
for di, d in enumerate(day_labels):
    for h in range(24):
        mask = (type_day_test == d) & (hours_test == h)
        if mask.sum() > 5:  # only if enough observations
            mape_matrix[di, h] = mape(y_test[mask], y_pred_adapt_best[mask])

fig, ax = plt.subplots(figsize=(16, 4))
im = ax.imshow(mape_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels])
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Day Type")
ax.set_title(f"MAPE (%) Heatmap — Best Adaptive Gating (threshold={best_threshold})")

# Add text annotations
for di in range(len(day_labels)):
    for h in range(24):
        if not np.isnan(mape_matrix[di, h]):
            color = "white" if mape_matrix[di, h] > np.nanmedian(mape_matrix) else "black"
            ax.text(h, di, f"{mape_matrix[di, h]:.1f}",
                    ha="center", va="center", fontsize=6, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("MAPE (%)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_mape_heatmap_adaptive.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Mejora de Adaptive Gating sobre Hard Gating por hora
# ============================================================

improvement_by_hour = {}
for h in range(24):
    mask_h = hours_test == h
    if mask_h.sum() > 0:
        mape_h_hard = mape(y_test[mask_h], y_pred_hard[mask_h])
        mape_h_adapt = mape(y_test[mask_h], y_pred_adapt_best[mask_h])
        improvement_by_hour[h] = mape_h_hard - mape_h_adapt  # positivo = adaptive es mejor

fig, ax = plt.subplots(figsize=(14, 5))
hours_list = sorted(improvement_by_hour.keys())
improvements = [improvement_by_hour[h] for h in hours_list]
colors = [PALETTE[3] if imp >= 0 else PALETTE[6] for imp in improvements]

ax.bar(hours_list, improvements, color=colors, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Mejora en MAPE (pp)")
ax.set_title(f"Mejora en MAPE: Adaptive (thr={best_threshold}) vs Hard Gating")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])

# Anotar
for h, imp in zip(hours_list, improvements):
    if abs(imp) > 0.01:
        va = "bottom" if imp >= 0 else "top"
        ax.text(h, imp, f"{imp:+.2f}", ha="center", va=va, fontsize=7)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_improvement_adaptive_vs_hard_by_hour.png",
            bbox_inches="tight")
plt.show()

# También por tipo de día
print("\nMejora en MAPE (pp) por tipo de día (positivo = adaptive mejor):")
for d in day_labels:
    mask_d = type_day_test == d
    if mask_d.sum() > 0:
        mape_d_hard = mape(y_test[mask_d], y_pred_hard[mask_d])
        mape_d_adapt = mape(y_test[mask_d], y_pred_adapt_best[mask_d])
        imp = mape_d_hard - mape_d_adapt
        name = DAY_TYPE_NAMES.get(d, f"Type {d}")
        print(f"  {name}: {imp:+.4f} pp  (Hard={mape_d_hard:.4f}%, Adapt={mape_d_adapt:.4f}%)")

In [ ]:
# ============================================================
# Improvement of adaptive over hard gating by hour
# ============================================================

improvement_by_hour = {}
for h in range(24):
    mask_h = hours_test == h
    if mask_h.sum() > 0:
        mape_h_hard = mape(y_test[mask_h], y_pred_hard[mask_h])
        mape_h_adapt = mape(y_test[mask_h], y_pred_adapt_best[mask_h])
        improvement_by_hour[h] = mape_h_hard - mape_h_adapt  # positive = adaptive is better

fig, ax = plt.subplots(figsize=(14, 5))
hours_list = sorted(improvement_by_hour.keys())
improvements = [improvement_by_hour[h] for h in hours_list]
colors = [PALETTE[3] if imp >= 0 else PALETTE[6] for imp in improvements]

ax.bar(hours_list, improvements, color=colors, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("MAPE Improvement (pp)")
ax.set_title(f"MAPE Improvement: Adaptive (thr={best_threshold}) vs Hard Gating")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])

# Annotate
for h, imp in zip(hours_list, improvements):
    if abs(imp) > 0.01:
        va = "bottom" if imp >= 0 else "top"
        ax.text(h, imp, f"{imp:+.2f}", ha="center", va=va, fontsize=7)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/06_improvement_adaptive_vs_hard_by_hour.png",
            bbox_inches="tight")
plt.show()

---
## 6. Guardar Resultados y la Mejor Configuración de Gating

Persistimos:
1. La tabla completa de comparación en formato CSV.
2. Los mejores parámetros de gating en formato JSON (consumidos por el Notebook 07 y el
   pipeline de producción).

In [ ]:
# ============================================================
# Guardar resultados de comparación de gating y mejor configuración
# ============================================================

# 1. Tabla completa de comparación
results_path = Path(RESULTS_DIR) / "06_gating_comparison.csv"
df_results.to_csv(results_path, index=False)
print(f"Comparación completa guardada en: {results_path}")

# 2. Mejor configuración por método
best_per_method_path = Path(RESULTS_DIR) / "06_best_per_method.csv"
best_per_method.to_csv(best_per_method_path, index=False)
print(f"Mejor por método guardado en: {best_per_method_path}")

# 3. Mejores parámetros globales de gating (JSON)
best_gating_config = {
    "gating_mode": "adaptive" if overall_best["method"] == "Adaptive" else overall_best["method"].lower().replace(" ", "_"),
    "best_method": overall_best["method"],
    "best_params": overall_best["params"],
    "best_mape_pct": float(overall_best["MAPE (%)"]),
    "best_rmse": float(overall_best["RMSE (kWh)"]),
    "threshold": float(best_threshold),
    "tau": float(best_tau),
    "min_prob": MIN_PROB,
    "pct_hard": float(overall_best["pct_hard"]),
    "pct_soft": float(overall_best["pct_soft"]),
    "hard_gating_mape": float(metrics_hard["MAPE (%)"]),
    "soft_distance_best_mape": float(df_dist.loc[best_tau_idx, "MAPE (%)"]),
    "soft_probability_mape": float(metrics_prob["MAPE (%)"]),
    "n_experts": len(experts),
    "expert_keys": [list(k) for k in sorted(experts.keys())],
}

config_path = Path(MODELS_DIR) / "gating" / "06_best_gating_config.json"
os.makedirs(config_path.parent, exist_ok=True)
with open(config_path, "w") as f:
    json.dump(best_gating_config, f, indent=2)
print(f"Mejor configuración de gating guardada en: {config_path}")

# 4. Resultados del barrido de threshold para Adaptive Gating
adaptive_path = Path(RESULTS_DIR) / "06_adaptive_threshold_sweep.csv"
df_adaptive.to_csv(adaptive_path, index=False)
print(f"Barrido adaptive guardado en: {adaptive_path}")

# 5. Resultados del barrido de tau para distancia
distance_path = Path(RESULTS_DIR) / "06_distance_tau_sweep.csv"
df_dist.to_csv(distance_path, index=False)
print(f"Barrido de distancia guardado en: {distance_path}")

print("\n" + "=" * 60)
print("RESUMEN")
print("=" * 60)
print(f"Mejor estrategia de gating: {overall_best['method']}")
print(f"Mejores parámetros: {overall_best['params']}")
print(f"MAPE global: {overall_best['MAPE (%)']:.4f}%")
print(f"Proporción Hard/Soft: {overall_best['pct_hard']:.1f}% / {overall_best['pct_soft']:.1f}%")
print(f"\nTodos los resultados se guardaron en: {RESULTS_DIR}/gating/")